# Final models: retrain on all labelled data

This notebook retrains the three members of our best ensemble (Exp1 ConvNeXt-base, Exp3 EfficientNetV2-S and Exp5 BiT ResNetV2-50x1; public score 0.98047) on all 12,260 labelled images, lowers Exp3's learning rate, chooses decision thresholds from held-out predictions where the data allow it, and writes new submission files. It is meant for one unattended **Run All** on the Mac.

The three trainings take about **6–7 hours** in total on the Mac GPU (estimates from earlier runs, see the timing table below). Each finished model is saved at once. If the run stops, a rerun skips finished models; if one model fails, the others still train and the ensemble falls back to that member's earlier fold-0 run, with a clear warning. Nothing is uploaded to Kaggle.

For a quick check before the overnight run, set `STAGE2_SMOKE_TEST=1` in the environment: it trains each model on a few batches, predicts 32 test images and writes nothing to `submissions/`. Its outputs carry a `_smoke` suffix and are not results.

## What changes, and why

| | Before | Problem | Now |
|---|---|---|---|
| **Training data** | Every model trained on folds 1–4; fold 0 (2,452 images) was only used for validation | The models never learned from 20% of the labels, including 27 of the 125 named-pet photos. The named pets decide 60% of macro F1. | Train on all 12,260 images: Moris 56, Motya 42 and Biatrix 41 photos instead of 46, 33 and 33. |
| **Training length** | The checkpoint with the best fold-0 macro F1 was kept, with early stopping | Without held-out data there is nothing to select on. Fold-0 selection was also optimistic, with 8–10 photos per pet. | A fixed number of epochs: the best epoch of each family's fold-0 run (Exp1 2, Exp3 8, Exp5 6), under the same 8-epoch learning-rate schedule, so each model sees the learning rates its best checkpoint had. The final weights are kept. |
| **Exp3 learning rate** | Backbone `1e-4`, head `1e-3`, ten times ConvNeXt's | Training loss fell to about zero while validation loss rose after epoch 4 (0.0055 to 0.0071), and the model made very confident mistakes. | Halved to backbone `5e-5` and head `5e-4`. We do not go down to BiT's `3e-5`: BiT was our weakest model on the leaderboard (0.94035) for reasons we have not tested, and the new rate cannot be validated tonight because no labelled data are held out. |
| **Head warm-up** | Each run trained its own head first (30–60 minutes) | Repeating it costs time and changes nothing for a final model | Each family starts from its fold-0 head. The head is only an initialization; reusing one trained on 80% of the same data does not leak anything into a model that is not evaluated on held-out data. |
| **Decision threshold** | 0.25 for every label, inherited from the logistic-regression baseline | Never checked for the fine-tuned models | Chosen per label from held-out fold-0 predictions, but only where there are enough positives (see the threshold section). We also keep a 0.25 version, so each change can be judged on its own. |
| **Running it** | Hard-coded flags, kernel state and overwrite errors cost several runs | | One Run All. Finished models are skipped on a rerun, a failure does not stop the others, and no file is overwritten. |

**Unchanged:** the saved folds and label order, 384-pixel mean-colour letterbox, horizontal flips as the only augmentation, BCEWithLogitsLoss, AdamW with weight decay 0.01 on weights only, effective batch 16, gradient clipping at 1.0, float32, and BatchNorm statistics frozen for EfficientNet. Exp1 was first trained on a Colab GPU; here it runs on the Mac GPU, so small numerical differences are expected.

**What we lose:** a model trained on all labels has no held-out score. The leaderboard is the only check of these models, the fixed epoch counts come from fold-0 runs, and the new Exp3 learning rate is untested.

In [1]:
from __future__ import annotations

import gc
import json
import os
import platform
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import torch

# The flags are set for one unattended Run All.
RUN_TRAINING = True  # Train the models that are not finished yet; finished ones are reused.
SMOKE_TEST = False  # A few batches per model; separate folders; no submission files.
ALLOW_WEIGHT_DOWNLOAD = True  # ConvNeXt's pretrained weights are not cached on this Mac yet.
USE_WANDB = not SMOKE_TEST  # Online logging as in the other runs; if W&B is unreachable, training continues with CSV logs.
BUILD_SUBMISSION = not SMOKE_TEST
SUBMIT_TO_KAGGLE = False  # No upload code exists; upload by hand.
if not ALLOW_WEIGHT_DOWNLOAD:
    os.environ['HF_HUB_OFFLINE'] = '1'  # Must be set before anything imports huggingface_hub.

SEED = 42
THRESHOLD = 0.25  # The fixed threshold of every earlier submission.
SCHEDULE_EPOCHS = 8  # One warm-up epoch, then cosine decay to 10% of the peak, over eight epochs, as in the fold-0 runs.
EFFECTIVE_BATCH_SIZE = 16
WEIGHT_DECAY = 1e-2
IMAGE_SIZE = 384
PRECISION = '32-true'
TEST_BATCH_SIZE = 8
EXPERIMENT_DIR = 'exp6'
WANDB_PROJECT = 'ai301-stage2'
MIN_POSITIVES_TO_TUNE = 100  # Tune a label's threshold only with at least this many held-out positives.
BEST_SUBMISSION = 'submissions/exp4_ensemble_exp1_exp3_exp5_mean_025.csv'  # Public 0.98047, for comparison.

# Each member: its settings for the full-data run and the fold-0 run it replaces.
MODELS = {
    'exp3': {
        'title': 'EfficientNetV2-S (Exp3), lower learning rate', 'run_id': 'full_effnetv2s_lowlr_seed42',
        'model_name': 'tf_efficientnetv2_s.in21k_ft_in1k', 'head': 'experiments/exp3/results/efficientnetv2s_fold0_seed42/head.pt',
        'backbone_lr': 5e-5, 'head_lr': 5e-4, 'epochs': 8, 'batch_size': 8, 'freeze_batchnorm': True,
        'fold0_run': 'efficientnetv2s_fold0_seed42', 'before': 'backbone 1e-4, head 1e-3; best fold-0 epoch 8 of 8',
        'estimate_hours': 2.3,
    },
    'exp5': {
        'title': 'BiT ResNetV2-50x1 (Exp5)', 'run_id': 'full_bit_r50x1_seed42',
        'model_name': 'resnetv2_50x1_bit.goog_in21k_ft_in1k', 'head': 'experiments/exp5/results/bit_r50x1_fold0_seed42/head.pt',
        'backbone_lr': 3e-5, 'head_lr': 3e-4, 'epochs': 6, 'batch_size': 8, 'freeze_batchnorm': False,
        'fold0_run': 'bit_r50x1_fold0_seed42', 'before': 'same learning rates; best fold-0 epoch 6 of 8',
        'estimate_hours': 1.5,
    },
    'exp1': {
        'title': 'ConvNeXt-base (Exp1)', 'run_id': 'full_convnext_base_seed42',
        'model_name': 'hf-hub:timm/convnext_base.fb_in22k_ft_in1k_384@2b2698d86523262ca8065d0957c7f80d963bb43f',
        'head': 'experiments/exp0/checkpoints/convnext_head_fold0_seed42_colab_head.pt',
        'backbone_lr': 1e-5, 'head_lr': 1e-4, 'epochs': 2, 'batch_size': 2, 'freeze_batchnorm': False,
        'fold0_run': 'convnext_standard_fold0_seed42_20261018_full1', 'before': 'same learning rates; best fold-0 epoch 2 of 8 (early stopping ended at 5)',
        'estimate_hours': 2.2,
    },
}
TRAIN_ORDER = ['exp3', 'exp5', 'exp1']  # ConvNeXt, the slowest, last: if the night is too short, only it falls back.

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}; smoke test: {SMOKE_TEST}; W&B: {USE_WANDB}; build submission: {BUILD_SUBMISSION}')

Device: mps; smoke test: False; W&B: True; build submission: True


## Data: every labelled image

We join `train.csv` to the saved folds as before and take all five folds for training. All 30,933 train and test images are decoded once before training, so a damaged file fails now and not at 3 a.m. The test images are predicted in `test.csv` order and written in `sample_submission.csv` order.

In [2]:
start = Path(os.environ.get('STAGE2_ROOT', Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents) if (path / 'folds.csv').is_file() and (path / 'stage2' / 'experiment.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this repository or a subdirectory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from stage2.batchnorm import FrozenBatchNormStatistics
from stage2.ensemble import load_finetuned_network, read_probabilities, write_test_probabilities
from stage2.experiment import LABEL_COLUMNS, ImageDataset, form_submission, preflight_images, read_csv, split_rows, verify_saved_stratification, write_run_record, write_submission
from stage2.finetune import FineTuneClassifier, StopAfterEpochs, load_trained_head
from stage2.run_records import file_sha256, record_failures
from stage2.thresholds import choose_thresholds
from stage2.train import predict_probabilities

DATA_DIR = ROOT / 'data'
verify_saved_stratification(DATA_DIR, ROOT / 'folds.csv', SEED)
folds_1_to_4, fold_0 = split_rows(DATA_DIR, ROOT / 'folds.csv', 0)
all_rows = folds_1_to_4 + fold_0
all_test_rows = read_csv(DATA_DIR / 'test.csv', ('image_id',))
all_test_ids = [row['image_id'] for row in all_test_rows]
named = ('Moris', 'Motya', 'Biatrix')
if SMOKE_TEST:
    pets = [row for row in all_rows if '1' in (row[label] for label in named)]
    train_rows = pets[:16] + [row for row in all_rows if row not in pets][:48]
    test_rows = all_test_rows[:32]
else:
    train_rows, test_rows = all_rows, all_test_rows
test_ids = [row['image_id'] for row in test_rows]
count = lambda rows: {label: sum(int(row[label]) for row in rows) for label in named}
print(f'Training images: {len(train_rows)} (before: {len(folds_1_to_4)}); named-pet photos {count(train_rows)} (before: {count(folds_1_to_4)})')
started = time.perf_counter()
preflight_images(train_rows, DATA_DIR / 'train')
preflight_images(test_rows, DATA_DIR / 'test')
print(f'Decoded {len(train_rows) + len(test_rows)} images in {time.perf_counter() - started:.0f} s; test images to predict: {len(test_rows)}')

Training images: 12260 (before: 9808); named-pet photos {'Moris': 56, 'Motya': 42, 'Biatrix': 41} (before: {'Moris': 46, 'Motya': 33, 'Biatrix': 33})
Decoded 30933 images in 25 s; test images to predict: 18673


## Check the pretrained weights, heads and logging before the long part

Each family is built once with its pretrained weights (downloaded if missing) and its fold-0 head, and must return five logits. The W&B key is looked up as in the other notebooks; if it is missing, the runs log to CSV only. Expected training time per model, scaled from the measured fold-0 epochs by the 25% larger training set:

In [3]:
import timm


def ensure_wandb_key() -> bool:
    key = os.environ.get('WANDB_API_KEY') or os.environ.get('WB_API_KEY')
    env_file = ROOT / '.env'
    if not key and env_file.is_file():
        for line in env_file.read_text().splitlines():
            name, separator, value = line.partition('=')
            if separator and name.strip() in {'WANDB_API_KEY', 'WB_API_KEY'} and value.strip().strip("\"'"):
                key = value.strip().strip("\"'")
                break
    if key:
        os.environ['WANDB_API_KEY'] = key
    return bool(key)


if USE_WANDB and not ensure_wandb_key():
    USE_WANDB = False
    print('No W&B key found; logging to CSV only.')
for name in TRAIN_ORDER:
    spec = MODELS[name]
    head_path = ROOT / spec['head']
    network = timm.create_model(spec['model_name'], pretrained=True, num_classes=len(LABEL_COLUMNS))
    load_trained_head(network, head_path)
    with torch.inference_mode():
        logits = network.eval()(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE))
    if logits.shape != (1, len(LABEL_COLUMNS)):
        raise ValueError(f'{name} returned {tuple(logits.shape)} logits.')
    spec['preprocessing'] = timm.data.resolve_model_data_config(network)
    print(f"{name}: {spec['title']} ready | {sum(p.numel() for p in network.parameters()) / 1e6:.1f}M parameters | "
          f"{spec['epochs']} epochs, backbone LR {spec['backbone_lr']}, head LR {spec['head_lr']} | about {spec['estimate_hours']} h")
    del network
print(f"Total estimate: about {sum(MODELS[name]['estimate_hours'] for name in TRAIN_ORDER):.1f} h, in the order {TRAIN_ORDER}.")

/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


exp3: EfficientNetV2-S (Exp3), lower learning rate ready | 20.2M parameters | 8 epochs, backbone LR 5e-05, head LR 0.0005 | about 2.3 h
exp5: BiT ResNetV2-50x1 (Exp5) ready | 23.5M parameters | 6 epochs, backbone LR 3e-05, head LR 0.0003 | about 1.5 h
exp1: ConvNeXt-base (Exp1) ready | 87.6M parameters | 2 epochs, backbone LR 1e-05, head LR 0.0001 | about 2.2 h
Total estimate: about 6.0 h, in the order ['exp3', 'exp5', 'exp1'].


## Train each model on all the data, then predict the test images

For each model, in the order EfficientNet, BiT, ConvNeXt:

1. If its run record, final weights and test predictions already exist (and the hashes match), it is reused. An unfinished attempt is moved aside to `<run-id>_incomplete_<time>`, not deleted.
2. The pretrained network gets its fold-0 head, every parameter is unfrozen, and it trains on all labelled images for its fixed number of epochs under the 8-epoch schedule. Each finished epoch prints its time and training loss.
3. The final weights are saved, then the 18,673 test images are predicted and saved as `<run-id>_test/`, in the same layout as the earlier runs. A single-model 0/1 CSV at 0.25 is written into the run folder (not into `submissions/`) in case you want its leaderboard score later.

Outputs go to `experiments/exp6/results/`. If a model fails, the error is printed and saved as `failure.json`, and the next model starts.

In [4]:
import lightning as L
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader

RESULTS = ROOT / 'experiments' / EXPERIMENT_DIR / 'results'
git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip() or None
source_files = ['notebooks/14_full_data_final.ipynb', 'stage2/experiment.py', 'stage2/train.py', 'stage2/finetune.py', 'stage2/batchnorm.py', 'stage2/ensemble.py', 'stage2/thresholds.py']
source_sha256 = {path: file_sha256(ROOT / path) for path in source_files}
versions = {'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'lightning': L.__version__}


def run_paths(spec: dict) -> dict:
    run_id = spec['run_id'] + ('_smoke' if SMOKE_TEST else '')
    run_dir, test_dir = RESULTS / run_id, RESULTS / f'{run_id}_test'
    return {'run_id': run_id, 'dir': run_dir, 'record': run_dir / f'{run_id}.json', 'checkpoint': run_dir / 'final.ckpt',
            'test_dir': test_dir, 'test_csv': test_dir / f'{run_id}_test.csv', 'test_json': test_dir / f'{run_id}_test.json'}


def completed_record(paths: dict) -> dict | None:
    if not (paths['record'].is_file() and paths['checkpoint'].is_file()):
        return None
    record = json.loads(paths['record'].read_text())
    if record.get('status') != 'completed' or file_sha256(paths['checkpoint']) != record['artifacts']['best_checkpoint_sha256']:
        return None
    return record


def finished(spec: dict) -> bool:
    paths = run_paths(spec)
    record = completed_record(paths)
    return (record is not None and paths['test_csv'].is_file() and paths['test_json'].is_file()
            and json.loads(paths['test_json'].read_text())['source_checkpoint_sha256'] == record['artifacts']['best_checkpoint_sha256'])


def set_aside(path: Path) -> None:
    if path.exists():
        aside = path.with_name(f'{path.name}_incomplete_{datetime.now():%Y%m%d_%H%M%S}')
        path.rename(aside)
        print(f'  moved an unfinished attempt to {aside.name}')


def free_memory() -> None:
    gc.collect()
    if DEVICE.type == 'mps':
        torch.mps.empty_cache()
    elif DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def train_full_data(name: str, spec: dict, paths: dict) -> dict:
    L.seed_everything(SEED, workers=True)
    network = timm.create_model(spec['model_name'], pretrained=True, num_classes=len(LABEL_COLUMNS))
    head_path = ROOT / spec['head']
    load_trained_head(network, head_path)
    for parameter in network.parameters():
        parameter.requires_grad_(True)
    network.train()
    mean, std = spec['preprocessing']['mean'], spec['preprocessing']['std']
    dataset = ImageDataset(train_rows, DATA_DIR / 'train', IMAGE_SIZE, mean, std, horizontal_flip=True)
    loader = DataLoader(dataset, batch_size=spec['batch_size'], shuffle=True, generator=torch.Generator().manual_seed(SEED), num_workers=0)
    module = FineTuneClassifier(network, spec['backbone_lr'], spec['head_lr'], WEIGHT_DECAY, THRESHOLD)
    accumulation = EFFECTIVE_BATCH_SIZE // spec['batch_size']
    epochs = 1 if SMOKE_TEST else spec['epochs']
    stopper = StopAfterEpochs(epochs)
    callbacks = [stopper, LearningRateMonitor(logging_interval='epoch')] + ([FrozenBatchNormStatistics()] if spec['freeze_batchnorm'] else [])
    loggers = [CSVLogger(save_dir=paths['dir'], name='curves')]
    wandb_logger = None
    if USE_WANDB:
        try:
            from lightning.pytorch.loggers import WandbLogger
            wandb_logger = WandbLogger(project=WANDB_PROJECT, name=paths['run_id'], save_dir=str(paths['dir']), log_model=False)
            wandb_logger.log_hyperparams({'model': spec['model_name'], 'training_data': 'all folds', 'epochs': epochs, 'schedule_epochs': SCHEDULE_EPOCHS,
                                          'backbone_lr': spec['backbone_lr'], 'head_lr': spec['head_lr'], 'batch_size': spec['batch_size'],
                                          'effective_batch_size': EFFECTIVE_BATCH_SIZE, 'image_size': IMAGE_SIZE, 'seed': SEED})
            loggers.append(wandb_logger)
        except Exception as error:
            print(f'  W&B unavailable ({type(error).__name__}); logging to CSV only.')
            wandb_logger = None
    trainer = L.Trainer(
        accelerator=DEVICE.type, devices=1, precision=PRECISION, max_epochs=SCHEDULE_EPOCHS, accumulate_grad_batches=accumulation,
        callbacks=callbacks, logger=loggers, limit_train_batches=2 * accumulation if SMOKE_TEST else 1.0, num_sanity_val_steps=0,
        gradient_clip_val=1.0, log_every_n_steps=5 if SMOKE_TEST else 50, enable_progress_bar=False, enable_checkpointing=False,
    )
    print(f"  training on {len(train_rows)} images for {epochs} epoch(s), {len(loader)} batches of {spec['batch_size']} per epoch", flush=True)
    started = time.perf_counter()
    try:
        trainer.fit(module, loader)
    finally:
        if wandb_logger is not None:
            try:
                wandb_logger.experiment.finish()
            except Exception:
                pass
    seconds = time.perf_counter() - started
    trainer.save_checkpoint(paths['checkpoint'], weights_only=True)
    record = {
        'run_id': paths['run_id'], 'experiment': '06_full_data', 'status': 'completed', 'fold': None, 'seed': SEED, 'smoke_test': SMOKE_TEST,
        'model': {'name': spec['model_name'], 'variant': 'standard', 'family': name, 'title': spec['title'],
                  'initial_head_checkpoint': spec['head'], 'initial_head_sha256': file_sha256(head_path), 'replaces_fold0_run': spec['fold0_run']},
        'data': {'training_images': len(train_rows), 'training_folds': 'all (0-4)', 'image_size': IMAGE_SIZE,
                 'transform': 'mean-colour letterbox, bicubic, checkpoint normalization; train horizontal flip p=0.5',
                 'fold_file': 'folds.csv', 'fold_sha256': file_sha256(ROOT / 'folds.csv'), 'label_columns': list(LABEL_COLUMNS), 'mean': list(mean), 'std': list(std)},
        'training': {'optimizer': f"AdamW(backbone_lr={spec['backbone_lr']}, head_lr={spec['head_lr']}, weight_decay={WEIGHT_DECAY} except bias/norm, betas=(0.9, 0.999))",
                     'scheduler': f'one epoch step warmup then cosine to 0.1 peak over {SCHEDULE_EPOCHS} epochs; stopped after {epochs}',
                     'loss': 'BCEWithLogitsLoss', 'epochs': epochs, 'physical_batch_size': spec['batch_size'], 'effective_batch_size': EFFECTIVE_BATCH_SIZE,
                     'precision': PRECISION, 'gradient_clip_norm': 1.0, 'batchnorm_statistics_frozen': spec['freeze_batchnorm'],
                     'checkpoint': 'final weights; no held-out data, so no selection', 'before': spec['before'],
                     'final_train_loss': float(trainer.callback_metrics['train_loss']) if 'train_loss' in trainer.callback_metrics else None},
        'timings': {'training_seconds': seconds, 'epoch_seconds': stopper.epoch_seconds},
        'artifacts': {'final_checkpoint': str(paths['checkpoint'].relative_to(ROOT)), 'best_checkpoint_sha256': file_sha256(paths['checkpoint']),
                      'curves_csv': str((Path(loggers[0].log_dir) / 'metrics.csv').relative_to(ROOT))},
        'hardware': {'device': str(DEVICE)}, 'versions': versions, 'git_commit': git_commit, 'source_sha256': source_sha256,
        'telemetry': {'wandb_run_id': wandb_logger.version if wandb_logger is not None else None},
        'public_leaderboard_score': None, 'error': None,
    }
    write_run_record(paths['record'], record)
    print(f'  trained in {seconds / 3600:.2f} h; saved {paths["checkpoint"].relative_to(ROOT)}', flush=True)
    del trainer, module, network, loader
    free_memory()
    return record


def predict_test(name: str, record: dict, paths: dict) -> None:
    set_aside(paths['test_dir'])
    network = load_finetuned_network(record, paths['checkpoint'])
    dataset = ImageDataset(test_rows, DATA_DIR / 'test', IMAGE_SIZE, record['data']['mean'], record['data']['std'])
    loader = DataLoader(dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=0)
    print(f'  predicting {len(test_rows)} test images...', flush=True)
    started = time.perf_counter()
    predicted_ids, probabilities = predict_probabilities(network, loader, DEVICE)
    seconds = time.perf_counter() - started
    if predicted_ids != test_ids:
        raise ValueError('The test loader changed the image order.')
    write_test_probabilities(paths['test_csv'], predicted_ids, probabilities)
    write_run_record(paths['test_json'], {
        'run_id': f"{paths['run_id']}_test", 'experiment': '06_full_data', 'status': 'test_predicted', 'source_run_id': paths['run_id'],
        'source_checkpoint_sha256': record['artifacts']['best_checkpoint_sha256'], 'test_images': len(predicted_ids),
        'transform': record['data']['transform'] + '; no test-time augmentation', 'device': str(DEVICE), 'prediction_seconds': seconds, 'versions': versions,
    })
    if not SMOKE_TEST:
        csv_path = paths['dir'] / f"{paths['run_id']}_{round(THRESHOLD * 100):03d}.csv"
        if not csv_path.exists():
            write_submission(csv_path, form_submission(DATA_DIR / 'sample_submission.csv', predicted_ids, probabilities, THRESHOLD))
    print(f'  test predictions saved in {seconds / 60:.1f} min', flush=True)
    del network
    free_memory()


def train_and_predict(name: str) -> str:
    spec, paths = MODELS[name], run_paths(MODELS[name])
    if finished(spec):
        print('  already finished; reusing its saved weights and test predictions.')
        return 'reused'
    record = completed_record(paths)
    if record is None:
        set_aside(paths['dir'])
        paths['dir'].mkdir(parents=True)
        configuration = {'run_id': paths['run_id'], 'experiment': '06_full_data', 'model_name': spec['model_name'], 'device': str(DEVICE), 'smoke_test': SMOKE_TEST}
        with record_failures(paths['dir'], configuration, 'training'):
            record = train_full_data(name, spec, paths)
    configuration = {'run_id': paths['run_id'], 'experiment': '06_full_data', 'model_name': spec['model_name'], 'device': str(DEVICE), 'smoke_test': SMOKE_TEST}
    with record_failures(paths['dir'], configuration, 'test_inference'):
        predict_test(name, record, paths)
    return 'trained and predicted'


outcomes = {}
if RUN_TRAINING:
    night_started = time.perf_counter()
    for name in TRAIN_ORDER:
        print(f"\n=== {name}: {MODELS[name]['title']} | started {datetime.now():%H:%M} ===", flush=True)
        try:
            outcomes[name] = train_and_predict(name)
        except Exception as error:
            outcomes[name] = f'FAILED ({type(error).__name__})'
            print(f'  {name} failed: {type(error).__name__}: {error}\n  Continuing with the next model; the ensemble will use its fold-0 run instead.', flush=True)
        finally:
            free_memory()
    print(f'\nAll models done in {(time.perf_counter() - night_started) / 3600:.2f} h:', outcomes)
else:
    print('Training disabled.')


=== exp3: EfficientNetV2-S (Exp3), lower learning rate | started 01:47 ===


Seed set to 42
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from WANDB_API_KEY.
wandb: Currently logged in as: hhavva (models-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..


  training on 12260 images for 8 epoch(s), 1533 batches of 8 per epoch


/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
Loading `train_dataloader` to estimate number of stepping batches.
/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

  | Name    | Type              | Params | Mode  | FLOPs
-------------------------------------------------------

...

  training on 12260 images for 2 epoch(s), 6130 batches of 2 per epoch


/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
Loading `train_dataloader` to estimate number of stepping batches.
/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

  | Name    | Type              | Params | Mode  | FLOPs
-------------------------------------------------------

  epoch 1/2: 45.4 min, train loss 0.00122
  epoch 2/2: 45.5 min, train loss 0.00039


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁████████████████
gradient_norm_backbone_epoch,█▁
gradient_norm_backbone_step,▂▂▁▁█▁▁▁▁▁█▁▂▂▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁
gradient_norm_epoch,█▁
gradient_norm_head_epoch,█▁
gradient_norm_head_step,▂▂▂▁▅▁▁▁▁▁█▁▁▂▁▁▁▁▁▄▁▁▁▁▁▁▁▁▁▁
gradient_norm_step,▂▂▁▁█▁▁▁▁▁█▁▂▂▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁
lr-AdamW/pg1,▁█
lr-AdamW/pg2,▁█
lr-AdamW/pg3,▁█
+3,...


  trained in 1.52 h; saved experiments/exp6/results/full_convnext_base_seed42/final.ckpt
  predicting 18673 test images...
  test predictions saved in 16.8 min

All models done in 5.37 h: {'exp3': 'trained and predicted', 'exp5': 'trained and predicted', 'exp1': 'trained and predicted'}


## Choose the decision thresholds

**How it was:** every submission used 0.25 for all five labels. That value was picked for the first baseline (logistic regression on frozen ConvNeXt features), where it beat 0.5 both in cross-validation and on the leaderboard because that model rarely gave the named pets a high probability. It was never re-checked for the fine-tuned CNNs, whose probabilities are mostly close to 0 or 1.

**Why it matters:** the threshold decides every image whose probability lies between 0 and 1. On the test set the Exp1 + Exp3 + Exp5 mean makes different decisions at 0.25 and at 0.5 in 156 label cells, more than the 81 cells that adding Exp5 changed.

**How we choose now:** thresholds need labelled images that the model did not train on. The full-data models have none, so we use the only held-out predictions these recipes have: the fold-0 predictions of the earlier fold-0 runs of the same three families, averaged as in the ensemble. For each label we compute F1 on a grid from 0.05 to 0.95, take every threshold that reaches the best F1 (the plateau), and use the middle of that plateau, which depends less on a single photo than either end. A bootstrap over the fold-0 images shows how stable that choice is.

**Only where there is enough evidence:** a label is tuned only if fold 0 has at least 100 positives. Cat and Dog have about 1,000 each. The named pets have 8–10, and only 5 of their fold-0 probabilities lie anywhere between 0.05 and 0.95: four real pets between 0.68 and 0.94, and one Biatrix false alarm at 0.29. Every threshold from 0.30 to 0.67 is then "perfect", so fold 0 cannot choose one; the bootstrap only looks stable because the same five photos decide every resample. The test set also seems to contain two to three times more named-pet photos than the training share, which argues against raising their threshold on such thin evidence. The named pets therefore keep 0.25.

**Even Cat and Dog are thin evidence.** Their choice is decided not by the 1,000 positives but by the few fold-0 images whose probability lies between 0.05 and 0.95: about 14 for Cat and 25 for Dog in the ensemble (the `between` column). The bootstrap range for Cat is wide (about 0.40 to 0.93). The tables for the single models are shown for the report only: there the choice rests on 5–10 images and is not used.

**Limits:** the thresholds come from the earlier models, and Exp3 now trains with a different learning rate, so its probabilities may shift. Cat and Dog cells are each worth about 0.00001 macro F1, so the tuned version is expected to differ from the 0.25 version only slightly. This is why the 0.25 file is the one to submit first.

In [5]:
fold0_targets = np.asarray([[int(row[label]) for label in LABEL_COLUMNS] for row in fold_0])
fold0_ids = [row['image_id'] for row in fold_0]
fold0_probabilities = {name: read_probabilities(ROOT / 'experiments' / name / 'results' / spec['fold0_run'] / f"{spec['fold0_run']}_val.csv", fold0_ids)
                       for name, spec in MODELS.items()}
fold0_mean = np.mean([fold0_probabilities[name] for name in MODELS], axis=0)


def print_choices(title: str, choices: list) -> None:
    print(title)
    print(f"  {'label':<8} {'positives':>9} {'between':>8} {'chosen':>7} {'tuned':>6} {'F1@0.25':>8} {'F1 chosen':>10} {'best plateau':>13} {'bootstrap 5-95%':>16}")
    for choice in choices:
        interval = f'{choice.bootstrap_interval[0]:.2f}-{choice.bootstrap_interval[1]:.2f}' if choice.bootstrap_interval else '-'
        print(f"  {choice.label:<8} {choice.positives:>9} {choice.informative:>8} {choice.threshold:>7.2f} {str(choice.tuned):>6} "
              f"{choice.f1_default:>8.4f} {choice.f1_chosen:>10.4f} {choice.plateau[0]:>6.2f}-{choice.plateau[1]:<6.2f} {interval:>16}")


ensemble_choices = choose_thresholds(fold0_targets, fold0_mean, default=THRESHOLD, min_positives=MIN_POSITIVES_TO_TUNE, bootstrap=200, seed=SEED)
print_choices('Ensemble (mean of the three fold-0 runs) -- these thresholds are used for the tuned submission:', ensemble_choices)
print("  ('between' = held-out probabilities strictly between 0.05 and 0.95, the only ones a threshold can affect)\n")
for name in MODELS:
    print_choices(f"{MODELS[name]['title']} alone (for the report only; with 5-10 images between 0.05 and 0.95 these choices are unreliable and not used):", choose_thresholds(fold0_targets, fold0_probabilities[name], default=THRESHOLD, min_positives=MIN_POSITIVES_TO_TUNE, bootstrap=0))
    print()
TUNED_THRESHOLDS = np.array([choice.threshold for choice in ensemble_choices])
before = np.mean([choice.f1_default for choice in ensemble_choices])
after = np.mean([choice.f1_chosen for choice in ensemble_choices])
print(f'Fold-0 macro F1 of the earlier ensemble: {before:.5f} at 0.25 -> {after:.5f} with the chosen thresholds '
      '(selected on the same images, so optimistic).')
print('Thresholds for the tuned submission:', dict(zip(LABEL_COLUMNS, TUNED_THRESHOLDS.round(2))))

Ensemble (mean of the three fold-0 runs) -- these thresholds are used for the tuned submission:
  label    positives  between  chosen  tuned  F1@0.25  F1 chosen  best plateau  bootstrap 5-95%
  Cat           1015       14    0.45   True   0.9956     0.9975   0.35-0.75          0.40-0.93
  Dog           1004       25    0.50   True   0.9960     0.9975   0.35-0.65          0.30-0.65
  Moris           10        2    0.25  False   1.0000     1.0000   0.05-0.80                  -
  Motya            9        2    0.25  False   1.0000     1.0000   0.05-0.65                  -
  Biatrix          8        1    0.25  False   0.9412     0.9412   0.30-0.95                  -
  ('between' = held-out probabilities strictly between 0.05 and 0.95, the only ones a threshold can affect)

EfficientNetV2-S (Exp3), lower learning rate alone (for the report only; with 5-10 images between 0.05 and 0.95 these choices are unreliable and not used):
  label    positives  between  chosen  tuned  F1@0.25  F1 chose

## Build the ensemble and the submission files

The ensemble is the equal-weight mean of the three members' test probabilities, as in the 0.98047 submission. If a full-data model did not finish, its earlier fold-0 run is used instead and the files get a `_partial` suffix, so they cannot be mistaken for the full result.

Two files are written to `submissions/`:

- **`exp6_full_data_ensemble_025.csv`**: threshold 0.25 for all labels. Compared with the 0.98047 file, only the models changed (all data, lower Exp3 learning rate). **Submit this one first**, so its score shows the effect of retraining on its own.
- **`exp6_full_data_ensemble_tuned.csv`**: the tuned thresholds (Cat and Dog from fold 0, named pets 0.25). Submit it second if you have a try left; its difference from the first file is the effect of the thresholds alone.

The cell prints how many test cells each file changes compared with the 0.98047 submission. Nothing is overwritten and nothing is uploaded.

In [6]:
row_of = {image_id: index for index, image_id in enumerate(all_test_ids)}
rows_now = [row_of[image_id] for image_id in test_ids]
member_probabilities, sources = {}, {}
for name, spec in MODELS.items():
    paths = run_paths(spec)
    if finished(spec):
        member_probabilities[name] = read_probabilities(paths['test_csv'], test_ids)
        sources[name] = f"full data ({paths['run_id']})"
    else:
        old_csv = ROOT / 'experiments' / name / 'results' / f"{spec['fold0_run']}_test" / f"{spec['fold0_run']}_test.csv"
        member_probabilities[name] = read_probabilities(old_csv, all_test_ids)[rows_now]
        sources[name] = f"FALLBACK: fold-0 run {spec['fold0_run']} (trained on 80% of the data)"
partial = any(source.startswith('FALLBACK') for source in sources.values())
for name, source in sources.items():
    print(f'{name}: {source}')
if partial:
    print('\n*** WARNING: not every member was retrained; the files are marked _partial. ***')

ensemble = np.mean([member_probabilities[name] for name in MODELS], axis=0)
decisions = {'0.25': ensemble >= THRESHOLD, 'tuned': ensemble >= TUNED_THRESHOLDS}
best_file = ROOT / BEST_SUBMISSION
best = read_probabilities(best_file, all_test_ids)[rows_now].astype(bool) if best_file.is_file() else None
print(f"\n{'labels':<8} {'best (0.98047)':>15} {'new @0.25':>10} {'new tuned':>10}")
for index, label in enumerate(LABEL_COLUMNS):
    best_count = int(best[:, index].sum()) if best is not None else -1
    print(f"{label:<8} {best_count:>15} {int(decisions['0.25'][:, index].sum()):>10} {int(decisions['tuned'][:, index].sum()):>10}")
if best is not None:
    for key, matrix in decisions.items():
        changed = matrix != best
        print(f'New ensemble at {key} differs from the 0.98047 submission in {int(changed.sum())} cells:', {label: int(changed[:, i].sum()) for i, label in enumerate(LABEL_COLUMNS)})
changed = decisions['0.25'] != decisions['tuned']
print('The two new files differ in', int(changed.sum()), 'cells:', {label: int(changed[:, i].sum()) for i, label in enumerate(LABEL_COLUMNS)})
for name in MODELS:
    old_csv = ROOT / 'experiments' / name / 'results' / f"{MODELS[name]['fold0_run']}_test" / f"{MODELS[name]['fold0_run']}_test.csv"
    if not sources[name].startswith('FALLBACK'):
        moved = (member_probabilities[name] >= THRESHOLD) != (read_probabilities(old_csv, all_test_ids)[rows_now] >= THRESHOLD)
        print(f'{name}: the full-data model differs from its fold-0 run in {int(moved.sum())} test cells ({int(moved[:, 2:].sum())} named-pet).')

if BUILD_SUBMISSION:
    suffix = '_partial' if partial else ''
    written = []
    for key, matrix in decisions.items():
        path = ROOT / 'submissions' / f"exp6_full_data_ensemble_{'025' if key == '0.25' else 'tuned'}{suffix}.csv"
        if path.exists():
            print(f'{path.name} already exists; not overwritten.')
            continue
        # form_submission thresholds probabilities; the 0/1 decisions pass through unchanged at 0.5.
        write_submission(path, form_submission(DATA_DIR / 'sample_submission.csv', test_ids, matrix.astype(float), 0.5))
        written.append(path)
    record_path = RESULTS / f'full_data_ensemble{suffix}' / f'full_data_ensemble{suffix}.json'
    if not record_path.exists():
        write_run_record(record_path, {
            'run_id': f'full_data_ensemble{suffix}', 'experiment': '06_full_data', 'status': 'completed',
            'method': 'equal-weight mean of member probabilities, then thresholds', 'members': sources,
            'thresholds': {'fixed': THRESHOLD, 'tuned': dict(zip(LABEL_COLUMNS, map(float, TUNED_THRESHOLDS)))},
            'threshold_choice': [choice.__dict__ for choice in ensemble_choices],
            'submissions': [str(path.relative_to(ROOT)) for path in written], 'git_commit': git_commit,
            'public_leaderboard_score': {'0.25': None, 'tuned': None},
        })
    for path in written:
        print('Ready for manual upload:', path.relative_to(ROOT))
else:
    print('No submission files written (smoke test or disabled).')

exp3: full data (full_effnetv2s_lowlr_seed42)
exp5: full data (full_bit_r50x1_seed42)
exp1: full data (full_convnext_base_seed42)

labels    best (0.98047)  new @0.25  new tuned
Cat                 7833       7792       7761
Dog                 7718       7751       7674
Moris                167        186        186
Motya                137        143        143
Biatrix              185        181        181
New ensemble at 0.25 differs from the 0.98047 submission in 189 cells: {'Cat': 59, 'Dog': 91, 'Moris': 21, 'Motya': 14, 'Biatrix': 4}
New ensemble at tuned differs from the 0.98047 submission in 183 cells: {'Cat': 74, 'Dog': 70, 'Moris': 21, 'Motya': 14, 'Biatrix': 4}
The two new files differ in 108 cells: {'Cat': 31, 'Dog': 77, 'Moris': 0, 'Motya': 0, 'Biatrix': 0}
exp3: the full-data model differs from its fold-0 run in 187 test cells (46 named-pet).
exp5: the full-data model differs from its fold-0 run in 326 test cells (65 named-pet).
exp1: the full-data model differs from its

## Result and conclusions

**Public score: `exp6_full_data_ensemble_025.csv` scored 0.97386, below the 0.98047 of the same three-model ensemble trained on folds 1–4.** Retraining on all labelled data did not help; it cost 0.0066 macro F1. That is far outside the 0.001–0.002 noise band estimated above, and it puts the new ensemble below the two-model Exp1 + Exp3 mean (0.97761) and barely above the best single models (Exp1 0.97003, Exp3 0.97015).

| Submission | Members | Threshold | Public score |
|---|---|---|---|
| `exp4_ensemble_exp1_exp3_exp5_mean_025.csv` | fold-0 runs (trained on folds 1–4, checkpoint chosen on fold 0) | 0.25 | **0.98047** |
| `exp6_full_data_ensemble_025.csv` | full-data runs (all folds, fixed epochs, final weights) | 0.25 | 0.97386 |

**Where the loss comes from.** The two files differ in 189 test cells. Cat and Dog account for 150 of them, but each such cell is worth about 0.00001, so together they explain at most about 0.0015. The rest of the drop, and so most of it, must come from the 39 named-pet cells. On the test set one named-pet photo is worth about 0.0007, so the drop matches roughly 9–10 more wrong named-pet decisions than before. The predictions show where they likely are:

| | 0.98047 | new @0.25 |
|---|---|---|
| Moris positives | 167 | 186 |
| Motya positives | 137 | 143 |
| Biatrix positives | 185 | 181 |
| Images predicted as both Moris and Motya | 28 | 42 |

- The 20 images that became Moris have a median ensemble probability of only 0.33; the old ensemble gave them 0.07. The new Motya positives look the same (median 0.36). These are weak, borderline additions.
- ConvNeXt drives them: it gives ≥ 0.5 on 14 of the 20 new Moris images, against 6 for EfficientNet and 5 for BiT. The full-data ConvNeXt is also less decisive on Moris: 125 test images have a Moris probability between 0.05 and 0.95, against 82 for its fold-0 run.
- Many new Moris + Motya pairs have one cat near 1.0 and the other at 0.3–0.45, which looks more like the two cats being confused than like real photos of both (10 of the 56 Moris training photos also show Motya, a lower share than in these predictions).

We have not checked these images by eye, so this is the most likely explanation, not a proven one.

**Why retraining may have hurt.** We changed several things at once, so the score cannot be attributed to one of them:

1. **No checkpoint selection.** The earlier members kept the epoch with the best fold-0 macro F1. Here the fixed epoch counts (2, 8, 6) were copied from those runs, but with only 8–10 photos per pet on fold 0, the "best epoch" was itself a noisy choice. It need not be a good stopping point for a model trained on different data. ConvNeXt in particular stops after 2 of 8 schedule epochs, close to its peak learning rate, so its final weights are taken mid-training.
2. **Exp3's learning rate was halved** without any held-out check. Its training loss still fell to 2e-5, so the change did not visibly calm the model.
3. **ConvNeXt was retrained on the Mac** instead of Colab. That alone should be a small numerical difference, but it adds to the confound.
4. **The extra 20% of data is small for the pets** (about 10 more photos each), so its expected gain was small. It was always going to be easy for the changes above to outweigh it.

The single-model 0.25 files in each run folder (for example `experiments/exp6/results/full_convnext_base_seed42/full_convnext_base_seed42_025.csv`) could show which member got worse, by comparing them with the fold-0 single scores (Exp1 0.97003, Exp3 0.97015, Exp5 0.94035). We have not submitted them.

**Tuned thresholds.** `exp6_full_data_ensemble_tuned.csv` differs from the 0.25 file only in 108 Cat and Dog cells, which can change the score by about 0.0015 at most. It cannot recover the 0.0066 loss, so we do not spend a submission on it.

**Decision.** The final submission stays `exp4_ensemble_exp1_exp3_exp5_mean_025.csv` (0.98047). The lesson is that once all labelled data go into training, there is no held-out set left to catch a regression. With so few named-pet photos, a recipe copied from a fold-0 run is not guaranteed to transfer, and the leaderboard became the only check. A safer way to use the extra data would have been one change at a time, for example retraining a single member while keeping its learning rate and comparing its single-model score, or training models on different folds and averaging them, so that each still has held-out predictions.